# SAP AI: Colab training

Select **Runtime → Change runtime type → T4 GPU**. Add `DATABASE_URL` under **Secrets** (key icon) and grant this notebook access. If the configured board export is absent, the notebook creates it from the database. It then reads a runtime-local copy and writes checkpoints and run artifacts to Drive.

Running all cells performs the smoke run by default. Full training is opt-in because it can take many hours and consume substantial Drive space.

In [ ]:
REPO_URL = "https://github.com/lgtyqz/sapai-python.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
DRIVE_RUN_DIR = "/content/drive/MyDrive/sapai-runs/run-001"  # @param {type:"string"}
BOARDS_JSONL = "/content/drive/MyDrive/sapai-data/boards.jsonl"  # @param {type:"string"}
BOARD_EXPORT_LIMIT = 10000  # @param {type:"integer"}
PACK = "Turtle"  # @param ["Turtle"]
SEED = 2026  # @param {type:"integer"}
REQUIRE_GPU = True  # @param {type:"boolean"}
RUN_FULL_TRAINING = False  # @param {type:"boolean"}
assert REPO_URL, "Set REPO_URL to the public Git repository containing this project."
assert BRANCH, "BRANCH cannot be empty."
assert DRIVE_RUN_DIR, "DRIVE_RUN_DIR cannot be empty."
assert BOARDS_JSONL, "BOARDS_JSONL cannot be empty."
assert 2 <= BOARD_EXPORT_LIMIT <= 10000, "BOARD_EXPORT_LIMIT must be between 2 and 10,000."

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

repo = Path('/content/sapai-python')
if repo.exists() and not (repo / '.git').is_dir():
    raise RuntimeError(f'{repo} exists but is not a Git checkout; remove or rename it.')
if not repo.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(repo)],
        check=True,
    )
else:
    origin = subprocess.run(
        ['git', '-C', str(repo), 'remote', 'get-url', 'origin'],
        check=True, capture_output=True, text=True,
    ).stdout.strip().removesuffix('/')
    if origin.removesuffix('.git') != REPO_URL.strip().removesuffix('/').removesuffix('.git'):
        raise RuntimeError(
            f'{repo} was cloned from {origin}, not {REPO_URL}. Restart the runtime or use a new REPO_URL.'
        )
    subprocess.run(['git', '-C', str(repo), 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '-B', BRANCH, 'FETCH_HEAD'], check=True)
os.chdir(repo)
assert sys.version_info >= (3, 11), f'Python 3.11+ is required, found {sys.version}'
GIT_COMMIT = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True
).stdout.strip()
print('Repository:', repo)
print('Commit:', GIT_COMMIT)

In [ ]:
%pip install -q -e '.[ml,neon,dev]'
%pip check

# Editable installs add a .pth file that a running Colab kernel may not reload.
# Add the src layout explicitly so this cell works without a runtime restart.
src_dir = str(repo / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
import importlib
importlib.invalidate_caches()
import sapai
print('sapai:', sapai.__file__)
assert Path(sapai.__file__).resolve().is_relative_to(repo.resolve()), (
    f'Imported sapai from the wrong location: {sapai.__file__}'
)

In [ ]:
import tensorflow as tf
from google.colab import userdata
from sapai.data.datasets import split_boards
from sapai.data.serialization import read_boards
from sapai.sim.battle import BattleSimulator
from sapai.sim.catalog import Catalog

gpus = tf.config.list_physical_devices('GPU')
print('Python:', sys.version.split()[0])
print('TensorFlow:', tf.__version__)
print('GPUs:', gpus)
if REQUIRE_GPU and not gpus:
    raise RuntimeError('No GPU is visible. Select Runtime → Change runtime type → T4 GPU, then restart.')

source_boards = Path(BOARDS_JSONL).expanduser()
if not source_boards.is_file():
    try:
        database_url = userdata.get('DATABASE_URL')
    except Exception as error:
        raise RuntimeError(
            'Could not read the DATABASE_URL Colab secret. Add it under Secrets and grant notebook access.'
        ) from error
    if not database_url:
        raise RuntimeError('The DATABASE_URL Colab secret is empty.')
    source_boards.parent.mkdir(parents=True, exist_ok=True)
    partial_boards = source_boards.with_name(source_boards.name + '.partial')
    export_environment = os.environ.copy()
    export_environment['DATABASE_URL'] = database_url
    export_result = subprocess.run([
        sys.executable, '-m', 'sapai.cli', 'export-boards',
        '--pack', PACK, '--limit', str(BOARD_EXPORT_LIMIT),
        '--output', str(partial_boards),
    ], check=False, env=export_environment, capture_output=True, text=True)
    if export_result.returncode != 0:
        diagnostic = '\n'.join(
            part.strip() for part in (export_result.stdout, export_result.stderr) if part.strip()
        ).replace(database_url, '[REDACTED DATABASE_URL]')
        raise RuntimeError(
            f'Board export failed with exit code {export_result.returncode}.\n{diagnostic}'
        )
    partial_boards.replace(source_boards)
    del database_url, export_environment
    print(f'Created stable board export: {source_boards}')
if source_boards.stat().st_size == 0:
    raise ValueError(f'Board dataset is empty: {source_boards}')
local_boards = Path('/content/sapai-data/boards.jsonl')
local_boards.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(source_boards, local_boards)
BOARDS_FOR_RUN = str(local_boards)

boards = [board for board in read_boards(local_boards) if board.pack == PACK]
if not boards:
    raise ValueError(f'{source_boards} contains no {PACK!r} boards.')
catalog = Catalog.from_json_dir(repo / 'assets' / 'data')
simulator = BattleSimulator(catalog)
for board in boards:
    simulator.assert_team_supported(board.team)
splits = split_boards(boards, seed=SEED)
for split_name in ('train', 'validation', 'test'):
    split = getattr(splits, split_name)
    groups = {}
    for board in split:
        key = (board.turn, board.pack, board.version)
        groups[key] = groups.get(key, 0) + 1
    if not any(count >= 2 for count in groups.values()):
        raise ValueError(
            f'{split_name} split cannot form a compatible battle pair. Add more replay groups.'
        )
print(f'Validated {len(boards):,} {PACK} boards; local snapshot: {local_boards}')
del boards, splits
Path(DRIVE_RUN_DIR).parent.mkdir(parents=True, exist_ok=True)
free_gib = shutil.disk_usage(Path(DRIVE_RUN_DIR).parent).free / 2**30
print(f'Drive free space: {free_gib:.1f} GiB')
if free_gib < 2:
    raise RuntimeError('Less than 2 GiB is free in Drive; choose another run directory or free space.')
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'sapai.cli', 'model-smoke'], check=True)

## Small end-to-end smoke run

This validates labeling, both checkpoint loops, complete Arena rollouts, and search distillation before a long run. It still needs enough replay IDs to form all three splits.

In [ ]:
smoke_dir = str(
    Path(DRIVE_RUN_DIR).with_name(Path(DRIVE_RUN_DIR).name + f'-smoke-{GIT_COMMIT[:8]}')
)
smoke = [
    sys.executable, '-m', 'sapai.cli', 'train-sequence',
    '--boards', BOARDS_FOR_RUN, '--workdir', smoke_dir, '--pack', PACK,
    '--battle-examples', '100', '--simulations-per-pair', '1',
    '--battle-epochs', '1', '--bootstrap-episodes', '2',
    '--bootstrap-epochs', '1', '--search-episodes', '1',
    '--search-epochs', '1', '--search-simulations', '4',
    '--search-candidates', '4', '--batch-size', '32', '--seed', str(SEED),
]
subprocess.run(smoke, check=True)

## Full training sequence

Edit counts here for the dataset and time budget. Rerunning resumes from Drive checkpoints.

In [ ]:
full = [
    sys.executable, '-m', 'sapai.cli', 'train-sequence',
    '--boards', BOARDS_FOR_RUN, '--workdir', DRIVE_RUN_DIR, '--pack', PACK,
    '--battle-examples', '100000', '--simulations-per-pair', '8',
    '--battle-epochs', '20', '--bootstrap-episodes', '1000',
    '--bootstrap-epochs', '20', '--search-episodes', '250',
    '--search-epochs', '5', '--search-simulations', '32',
    '--search-candidates', '8', '--batch-size', '128', '--seed', str(SEED),
]
if RUN_FULL_TRAINING:
    subprocess.run(full, check=True)
else:
    print('Set RUN_FULL_TRAINING=True when the smoke run succeeds.')

## Visualize the latest policy

The generated HTML contains both shops and battles and is also saved with the run artifacts.

In [ ]:
active_run = DRIVE_RUN_DIR if RUN_FULL_TRAINING else smoke_dir
visualization = str(Path(active_run) / 'arena.html')
subprocess.run([
    sys.executable, '-m', 'sapai.cli', 'visualize-arena',
    '--boards', BOARDS_FOR_RUN, '--pack', PACK, '--policy', 'model',
    '--policy-weights', str(Path(active_run) / 'policy-model'),
    '--seed', str(SEED), '--output', visualization,
], check=True)
archive = Path(active_run) / 'arena-visualization.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for name in ('arena.html', 'sapai.css', 'sapai.js'):
        bundle.write(Path(active_run) / name, name)
    for asset in (Path(active_run) / 'sapai-assets').rglob('*'):
        if asset.is_file():
            bundle.write(asset, asset.relative_to(active_run))
print(f'Visualization: {visualization}')
print(f'Portable archive: {archive}')
print('Open or download it from the Colab Files pane. Inline display is omitted because the HTML uses sibling CSS, JavaScript, and sprite files.')